# SearchLite-RL: full run on Colab
Runtime > Change runtime type > GPU (L4/A100). Run cells in order. Cell 2 restarts the runtime; then continue from cell 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!if [ -d Pilot-project ]; then cd Pilot-project && git pull; else git clone https://github.com/aasritha-oql7872/Pilot-project.git; fi
%cd /content/Pilot-project
!pip install -q -r requirements.txt
!pip uninstall -y -q torchvision torchaudio torchao
import os; os.kill(os.getpid(), 9)  # restart runtime so the new packages load

In [ ]:
%cd /content/Pilot-project
!nvidia-smi | head -12
!python -c "import torch,transformers,trl,peft;print(torch.__version__,transformers.__version__,trl.__version__,peft.__version__)"

In [ ]:
# Full pipeline (~35-40 min). Safe to leave running.
!rm -rf outputs
!bash run_all.sh 2>&1 | tee run_all.log | tail -150

In [ ]:
# Save results to Drive (adapters included, they are ~4 MB each)
!mkdir -p /content/drive/MyDrive/SearchLite-RL
!rm -rf /content/drive/MyDrive/SearchLite-RL/outputs
!cp -r outputs /content/drive/MyDrive/SearchLite-RL/outputs
!cp run_all.log /content/drive/MyDrive/SearchLite-RL/
!ls /content/drive/MyDrive/SearchLite-RL/outputs

In [ ]:
# Key numbers to paste back into the chat
!python -m src.compare --base outputs/main_v2/eval_base.json --trained outputs/main_v2/eval_trained.json outputs/seed43/eval_trained.json outputs/seed44/eval_trained.json
!python -m src.compare --base outputs/main_v2/eval_twohop_base.json --trained outputs/main_v2/eval_twohop_trained.json outputs/seed43/eval_twohop_trained.json outputs/seed44/eval_twohop_trained.json
!python -m src.compare --base outputs/mixed/eval_twohop_base.json --trained outputs/mixed/eval_twohop_onehop.json outputs/mixed/eval_twohop_mixed.json
!python -m src.compare --base outputs/mixed/eval_test_base.json --trained outputs/mixed/eval_test_onehop.json outputs/mixed/eval_test_mixed.json
!cut -d, -f1,2 outputs/mixed/train_metrics.csv | head -3; python - <<'EOF'
import pandas as pd
d=pd.read_csv('outputs/mixed/train_metrics.csv')
print(d[['step','reward','searchlite/correct','searchlite/searches_per_episode','searchlite/invalid_per_episode']].iloc[::10].to_string())
EOF